# 🎬 AI Shorts Maker (Colab)
Free video factory. The code lives on GitHub (edit it with **Claude Code**), this notebook always pulls the newest version and makes the videos.

**How to use on a phone:** fill in the settings below, then menu **Runtime → Run all**. The finished video is saved to your Google Drive (folder `ShortsMaker`) and offered for download.

First time only: the notebook asks for your free **Gemini API key** (get it at aistudio.google.com → *Get API key*). Tip: save it in Colab **Secrets** (🔑 icon on the left) as `GEMINI_API_KEY` so you never have to type it again.

In [ ]:
#@title ⚙️ Settings
REPO_URL = "https://github.com/YOUR-NAME/shorts-maker"  #@param {type:"string"}
GENRE = "horror"  #@param ["horror", "funny", "motivational", "mystery", "fantasy", "scifi", "wholesome"] {allow-input: true}
COUNT = 1  #@param {type:"slider", min:1, max:5, step:1}
CUSTOM_STYLE = ""  #@param {type:"string"}
SAVE_TO_DRIVE = True  #@param {type:"boolean"}
VOICE = "en-US-ChristopherNeural"  #@param ["en-US-ChristopherNeural", "en-US-GuyNeural", "en-US-AriaNeural", "en-US-JennyNeural", "en-GB-RyanNeural", "en-GB-SoniaNeural"]

In [ ]:
#@title 📥 Get newest code + install
import os, subprocess
from getpass import getpass

def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return os.environ.get(name, "")

url = REPO_URL.strip().rstrip("/")
if url.endswith(".git"): url = url[:-4]
token = secret("GITHUB_TOKEN")   # only needed if the repo is private
clone_url = url.replace("https://", f"https://{token}@") if token else url

if os.path.isdir("/content/repo/.git"):
    subprocess.run(["git", "-C", "/content/repo", "pull", "-q"], check=True)
else:
    r = subprocess.run(["git", "clone", "-q", clone_url + ".git", "/content/repo"])
    if r.returncode != 0:
        raise SystemExit("Could not download the repo. Check REPO_URL, or if it is private "
                         "add a GitHub token as Colab secret GITHUB_TOKEN.")
%cd /content/repo
!pip install -q -r requirements.txt
!apt-get -qq install -y fonts-liberation > /dev/null 2>&1 || true
print("✅ Code ready")

In [ ]:
#@title 🔑 API keys (from Colab Secrets, or asked once)
for name, needed in [("GEMINI_API_KEY", True), ("GROQ_API_KEY", False), ("YOUTUBE_API_KEY", False),
                     ("ANTHROPIC_API_KEY", False), ("OPENAI_API_KEY", False)]:
    val = secret(name)
    if not val and needed and not any(secret(k) for k in ("GROQ_API_KEY", "ANTHROPIC_API_KEY", "OPENAI_API_KEY")):
        val = getpass(f"Paste your {name} (free at aistudio.google.com): ").strip()
    if val:
        os.environ[name] = val
        print(f"✅ {name}")
os.environ["TTS_VOICE"] = VOICE
if not os.environ.get("YOUTUBE_API_KEY"):
    print("ℹ️ No YOUTUBE_API_KEY - stories still work, just without live trend data.")

In [ ]:
#@title 💾 Where to save
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    base = "/content/drive/MyDrive/ShortsMaker"
    os.makedirs(base + "/music", exist_ok=True)
    os.environ["SHORTS_OUTPUT_DIR"] = base + "/output"
    os.environ["SHORTS_HISTORY_FILE"] = base + "/history.json"   # remembers used ideas
    os.environ["SHORTS_MUSIC_DIR"] = base + "/music"             # put royalty-free mp3s here
    print("✅ Saving to Google Drive → ShortsMaker")
else:
    print("Saving only inside Colab (lost when it closes) - download the video below.")

In [ ]:
#@title 🎬 Make the videos
args = ["python", "main.py", "make", "--genre", GENRE, "--count", str(COUNT)]
if CUSTOM_STYLE.strip():
    args += ["--style", CUSTOM_STYLE.strip()]
import shlex
cmd = " ".join(shlex.quote(a) for a in args)
!{cmd}

In [ ]:
#@title ▶️ Watch + download the newest video
import glob
from IPython.display import HTML, display
from base64 import b64encode
out_dir = os.environ.get("SHORTS_OUTPUT_DIR", "/content/repo/output")
videos = sorted(glob.glob(out_dir + "/*/short.mp4"), key=os.path.getmtime)
if not videos:
    print("No video found - look for an error in the step above.")
else:
    latest = videos[-1]
    info = os.path.join(os.path.dirname(latest), "upload_info.txt")
    print(open(info, encoding="utf-8").read())
    data = b64encode(open(latest, "rb").read()).decode()
    display(HTML(f'<video width="300" controls playsinline src="data:video/mp4;base64,{data}"></video>'))
    try:
        from google.colab import files
        files.download(latest)
    except Exception:
        pass
    print("📱 On iPhone the easiest way: open the Google Drive app → ShortsMaker → output → "
          "tap the video → ⋯ → Send a copy → Save Video.")